# Optimización de Hiperparámetros

En nuestro modelo anterior, logramos un RMSE de $47.328 con un modelo RandomForestRegressor. Vamos a ajustar sus hiperparámetros para intentar reducir aún más este error.

## Pasos Preliminares

### Importación del Pipeline de Preprocesamiento

El pipeline de preprocesamiento desarrollado en notebooks anteriores se importa desde el módulo compartido [`utils/housing_preprocessing.py`](utils/housing_preprocessing.py). Usamos un valor por defecto bajo para `n_clusters` ya que vamos a ajustar este hiperparámetro.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from utils.housing_preprocessing import get_preprocessing_pipeline
preprocessing = get_preprocessing_pipeline(n_clusters=10)  # Valor bajo por defecto, será ajustado

In [ ]:
full_pipeline = Pipeline([    ("preprocessing", preprocessing),    ("random_forest", RandomForestRegressor(random_state=42)),])

### Carga de Datos

La carga de datos con la división estratificada entrenamiento/prueba se importa desde [`utils/load_california.py`](utils/load_california.py).

In [ ]:
from utils.load_california import load_housing_dataX_train, X_test, y_train, y_test = load_housing_data()

## Visualización de Hiperparámetros

In [ ]:
full_pipeline

,steps,"[('preprocessing', ...), ('random_forest', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bedrooms', ...), ('rooms_per_house', ...), ...]"
,remainder,Pipeline(step...ardScaler())])
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Para ver los nombres de los hiperparámetros que pueden ajustarse, puede usarse el siguiente código:

In [ ]:
for param in sorted(full_pipeline.get_params().keys()):
    print(param)

## *Búsqueda en malla* (*Grid Search*)

Para evitar el tedioso proceso de modificar manualmente los hiperparámetros de un modelo hasta encontrar los que producen los mejores resultados, podemos definir todos los valores de hiperparámetros que queremos probar y programarlos para intentar todas las combinaciones posibles.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {'preprocessing__geo__n_clusters': [5, 8, 10], # número de clústeres para el transformador geográfico
     'random_forest__max_features': [4, 6, 8]}, # número de características a considerar al buscar la mejor división
    {'preprocessing__geo__n_clusters': [10, 15],
     'random_forest__max_features': [6, 8, 10]},
]

grid_search = GridSearchCV(
    estimator = full_pipeline,
    param_grid = param_grid, 
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
    )
_ = grid_search.fit(X_train, y_train)

El parámetro `param_grid` es una lista de diccionarios, cada uno con los valores de hiperparámetros que queremos probar. En este caso, primero probamos 3 valores para el número de clústeres y 3 para el número de características consideradas en cada división. Luego probamos 2 valores para el número de clústeres y 3 para el número de características. En total, probamos 3×3 + 2×3 = 15 combinaciones de hiperparámetros.

Además, el parámetro `n_jobs` permite paralelizar la búsqueda de hiperparámetros indicando el número de procesadores a usar; un valor de -1 significa que se usarán todos los procesadores disponibles. Este mismo parámetro puede usarse en el modelo RandomForestRegressor para paralelizar la construcción de árboles, pero hay que tener cuidado si se hace ambas cosas, ya que si se paraleliza cada búsqueda de hiperparámetros, que es en sí misma una ejecución del modelo, y ese modelo a su vez paraleliza la construcción de árboles, el número total de ejecuciones se multiplicaría. El total de `n_jobs` de RandomForestRegressor multiplicado por el número de búsquedas no puede superar el número de núcleos físicos de la máquina. En general, es mejor paralelizar la búsqueda de hiperparámetros en lugar de la construcción de árboles; por ello, en este caso hemos optado por dejar RandomForestRegressor con el valor por defecto `n_jobs=None`, que asigna 1 núcleo por árbol, y usar el máximo para GridSearchCV con `n_jobs=-1`.

Ahora podemos ver los mejores hiperparámetros encontrados.

In [ ]:
grid_search.best_params_

{'preprocessing__geo__n_clusters': 15, 'random_forest__max_features': 6}

Podemos ver que el mejor modelo tiene 15 clústeres. Como este es el valor más alto probado, tendría sentido ejecutar nuevas pruebas con valores mayores.

También devuelve el mejor estimador encontrado:

In [ ]:
grid_search.best_estimator_

,steps,"[('preprocessing', ...), ('random_forest', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bedrooms', ...), ('rooms_per_house', ...), ...]"
,remainder,Pipeline(step...ardScaler())])
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


También podemos ver el resultado de cada combinación de hiperparámetros probada durante la búsqueda:

In [ ]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
# Seleccionar las columnas que queremos mostrar
cv_res = cv_res[["param_preprocessing__geo__n_clusters",
                 "param_random_forest__max_features", "split0_test_score",
                 "split1_test_score", "split2_test_score", "mean_test_score"]]
# Renombrar columnas para mayor claridad
score_cols = ["split0", "split1", "split2", "mean_test_rmse"]
cv_res.columns = ["n_clusters", "max_features"] + score_cols
# Limpiar la métrica de puntuación
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)
cv_res.head()

## Búsqueda Aleatoria (*Randomized Search*)

En lugar de probar todas las combinaciones posibles de hiperparámetros, RandomizedSearchCV permite probar un número especificado de combinaciones aleatorias. Esto ofrece ciertas ventajas:

- **Eficiencia computacional**: La "maldición de la dimensionalidad" hace que la búsqueda en malla sea computacionalmente inviable muy rápidamente. Con más de 3 o 4 hiperparámetros con varias opciones cada uno, el número total de combinaciones a probar explota. La búsqueda aleatoria permite fijar un presupuesto computacional (número de iteraciones) independiente del número de hiperparámetros, haciéndola viable para problemas complejos.
- **Efectividad en altas dimensiones**: Para muchas funciones objetivo (como el rendimiento del modelo), solo unos pocos hiperparámetros tienen un impacto significativo. Al muestrear aleatoriamente combinaciones, hay mayor probabilidad de probar valores diversos en las dimensiones importantes, mientras que la búsqueda en malla desperdicia mucho esfuerzo probando valores sistemáticamente en dimensiones que apenas afectan al resultado.
- **Manejo de parámetros continuos**: La búsqueda aleatoria maneja de forma natural los parámetros continuos muestreando de una distribución (por ejemplo, uniforme, log-uniforme). La búsqueda en malla requiere discretizar el rango, lo cual es artificial y puede fácilmente pasar por alto el valor óptimo real si cae entre los puntos de la malla.

Por estas razones, RandomizedSearchCV es generalmente más eficiente que GridSearchCV para problemas de alta dimensionalidad, con muchos hiperparámetros, o cuando no tenemos una idea clara de los rangos de valores de los hiperparámetros.

In [ ]:
list(range(3,50))

[3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49]

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distribs = {
    'preprocessing__geo__n_clusters': randint(low=3, high=50),
    'random_forest__max_features': randint(low=2, high=20)
    }

rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distribs,
    n_iter=10, # número de iteraciones
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
    )
_ = rnd_search.fit(X_train, y_train)

`scipy.stats.randint()` devuelve un objeto que contiene la distribución de probabilidad de la variable aleatoria discreta. RandomizedSearchCV lo usa para muestrear aleatoriamente valores de hiperparámetros.

El parámetro `n_iter` es el número de iteraciones a realizar. En este caso, probamos 10 combinaciones de hiperparámetros.

In [ ]:
cv_res = pd.DataFrame(rnd_search.cv_results_)cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)cv_res = cv_res[["param_preprocessing__geo__n_clusters",                 "param_random_forest__max_features", "split0_test_score",                 "split1_test_score", "split2_test_score", "mean_test_score"]]score_cols = ["split0", "split1", "split2", "mean_test_rmse"]cv_res.columns = ["n_clusters", "max_features"] + score_colscv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)cv_res.head()

,n_clusters,max_features,split0,split1,split2,mean_test_rmse
1,45,9,41639,42971,43353,42654
8,32,7,41883,43500,43574,42986
5,42,4,41922,44248,43561,43244
6,24,3,42190,44228,43890,43436
0,41,16,42547,43497,44434,43492


Hemos logrado mejorar nuestro modelo reduciendo el RMSE a $42.560 definiendo 45 clústeres y considerando 9 características para cada división.

## Evaluación del Modelo Final en el Conjunto de Prueba

In [ ]:
from sklearn.metrics import root_mean_squared_errorfinal_predictions = rnd_search.best_estimator_.predict(X_test)final_rmse = root_mean_squared_error(y_test, final_predictions)print(final_rmse)

39779.37181076267
